# **Install & Imports**

In [1]:
# Install the libraries we need
%pip install -q peft==0.4.0

ERROR: Could not find a version that satisfies the requirement torch>=1.13.0 (from peft) (from versions: none)
ERROR: No matching distribution found for torch>=1.13.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install datasets

In [3]:
# Imports
import os, time
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

ModuleNotFoundError: No module named 'torch'

# **Load the base model + tokenizer**

In [ ]:
# Choose a small causal LLM to keep things light
model_name = "bigscience/bloomz-560m"

# Tokenizer turns text ↔ tokens
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
# BLOOMZ needs a pad token for training sometimes; reuse eos if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Foundation model (CAUSAL LM = next-token prediction)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
foundation_model.config.pad_token_id = tokenizer.pad_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

# **Load the dataset (10% sample) and tokenize**

In [ ]:
# Load quotes (train split only) and take a small 10% sample
data = load_dataset("Abirate/english_quotes", split="train")
data = data.shuffle(seed=42).select(range(max(1, len(data)//10)))  # ~10%

# Tokenize each "quote" string into input IDs
# We keep it super simple: pack single quotes, truncate long ones
def tokenize_example(batch):
    return tokenizer(
        batch["quote"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )

tokenized = data.map(tokenize_example, batched=True, remove_columns=data.column_names)

# Tiny train subset for a quick demo (you can increase later)
train_sample = tokenized.select(range(min(256, len(tokenized))))
train_sample

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 250
})

# **Configure LoRA (what to adapt)**

In [ ]:
# LoRA config:
# - r: rank (size of the tiny adapter bottleneck)
# - alpha: scaling factor (often 1–32)
# - target_modules: which Linear layers to adapt inside BLOOMZ
# - dropout: small regularization on adapter path
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# **Wrap the base model with LoRA adapters**

In [ ]:
# Attach LoRA adapters to the frozen base model
peft_model = get_peft_model(foundation_model, lora_config)

# Confirm we’re only training a tiny fraction of params
peft_model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 562,360,320 || trainable%: 0.5593794384354857


# **Build data collator (for causal LM)**

In [ ]:
# Collator that creates causal-LM batches (no MLM)
collator = transformers.DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# **Training arguments (simple & CPU-friendly)**

In [ ]:
output_directory = os.path.join("./cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,     # helps on small machines
    learning_rate=3e-2,            # LoRA can use higher LR than full FT
    num_train_epochs=1,            # quick demo; bump to 2–3 if you want
    fp16=False,                    # set True if on GPU with FP16
    bf16=False,                    # set True if on BF16 GPU
    logging_steps=10,
    save_strategy="no",
    evaluation_strategy="no",
    per_device_train_batch_size=4,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

# **Initialize Trainer and train**

In [ ]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=collator,
)
trainer.train()

# **Save the LoRA adapter**

In [ ]:
# Save only the adapter (tiny!) so we can reuse the same base model
time_now = time.strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

peft_model_path

# **Reload for inference (base model + adapter)**

In [ ]:
# Fresh base model
base_for_infer = AutoModelForCausalLM.from_pretrained(model_name)
base_for_infer.config.pad_token_id = tokenizer.pad_token_id

# Attach saved LoRA adapter (inference only)
infer_model = PeftModel.from_pretrained(
    base_for_infer,
    peft_model_path,
    is_trainable=False
)
infer_model.eval()

# **Generate text**

In [ ]:
# Prompt the model to continue a famous quote
prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = infer_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,       # sampling for more variety
        temperature=0.9,      # a little creativity
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])